In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
real_df=pd.read_csv("/kaggle/input/datasets/toshangupta/taiwan-credit-card-dataset/data.csv")

syn_df=pd.read_csv("/kaggle/input/datasets/toshangupta/synthetic-gan-without-dp/synth_dp.csv")
#syn_df=syn_df.drop(columns=["Unnamed: 0"])
syn_df = syn_df[real_df.columns]


# Split real data
real_train, real_test = train_test_split(
    real_df,
    test_size=0.2,
    random_state=42
)

In [2]:
categorical_cols=["SEX", "EDUCATION", "MARRIAGE","PAY_0", "PAY_2","PAY_3", "PAY_4", "PAY_5", "PAY_6","default"]


In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, r2_score
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMClassifier, LGBMRegressor


def compute_structural_fidelity_tabstruct(
    real_df,
    synthetic_df,
    test_size=0.2,
    random_state=42,
):
    feature_scores = {}
    for target in real_df.columns:
        print(target)
        X_train_syn = synthetic_df.drop(columns=[target])
        y_train_syn = synthetic_df[target]

        X_test_real = real_df.drop(columns=[target])
        y_test_real = real_df[target]
        if target in categorical_cols:
            # Classification
            model = LGBMClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=-1,
                random_state=random_state,
            )

            model.fit(X_train_syn, y_train_syn)
            y_pred = model.predict(X_test_real)
            score = accuracy_score(y_test_real, y_pred)

        else:
            # Regression
            model = LGBMRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=-1,
                random_state=random_state,
            )

            model.fit(X_train_syn, y_train_syn)
            y_pred = model.predict(X_test_real)
            score = np.sqrt(mean_squared_error(y_test_real, y_pred))
        feature_scores[target] = score
    return feature_scores


In [4]:
def compute_utilities(per_feature, per_feature_real, target_variable):
    per_feature_utility = {}
    for feature in per_feature:
        synth_perf = per_feature[feature]
        real_perf = per_feature_real[feature]

        if feature in categorical_cols:
            utility = synth_perf / real_perf
        else:
            utility = real_perf / synth_perf
        per_feature_utility[feature] = utility
    global_utility = sum(per_feature_utility.values()) / len(per_feature_utility)
    local_utility = per_feature_utility[target_variable]
    return global_utility, local_utility, per_feature_utility

In [5]:
perf_syn= compute_structural_fidelity_tabstruct(
    real_test,
    syn_df
)

perf_real= compute_structural_fidelity_tabstruct(
    real_test,
    real_train
)

LIMIT_BAL
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010124 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3129
[LightGBM] [Info] Number of data points in the train set: 30000, number of used features: 23
[LightGBM] [Info] Start training from score 140970.000000
SEX
[LightGBM] [Info] Number of positive: 18196, number of negative: 11804
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001266 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3155
[LightGBM] [Info] Number of data points in the train set: 30000, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.606533 -> initscore=0.432763
[LightGBM] [Info] Start training from score 0.432763
EDUCATION
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of

In [6]:
print("Performance Scores Synthetic:", perf_syn)
print("Performance Scores Real:", perf_real)

Performance Scores Synthetic: {'LIMIT_BAL': np.float64(113417.63405571852), 'SEX': 0.6035, 'EDUCATION': 0.4831666666666667, 'MARRIAGE': 0.6565, 'AGE': np.float64(9.49854997974385), 'PAY_0': 0.7996666666666666, 'PAY_2': 0.9111666666666667, 'PAY_3': 0.8851666666666667, 'PAY_4': 0.885, 'PAY_5': 0.8888333333333334, 'PAY_6': 0.8501666666666666, 'BILL_AMT1': np.float64(27137.41340829437), 'BILL_AMT2': np.float64(19932.522075181183), 'BILL_AMT3': np.float64(18749.679652580475), 'BILL_AMT4': np.float64(18536.473744929222), 'BILL_AMT5': np.float64(17450.969796011483), 'BILL_AMT6': np.float64(20488.471924098412), 'PAY_AMT1': np.float64(14002.305750919815), 'PAY_AMT2': np.float64(13390.713560778602), 'PAY_AMT3': np.float64(18846.188846661087), 'PAY_AMT4': np.float64(17597.218221569216), 'PAY_AMT5': np.float64(16404.70155631371), 'PAY_AMT6': np.float64(21732.58272116536), 'default': 0.7958333333333333}
Performance Scores Real: {'LIMIT_BAL': np.float64(92277.17190629602), 'SEX': 0.6383333333333333,

In [7]:
global_utility, local_utility, per_feature_utility=compute_utilities(perf_syn, perf_real, "default")
print(f"Global Utility: {global_utility}")
print(f"Local Utility: {local_utility}")
print(f"Per Feature Utility: {per_feature_utility}")

Global Utility: 0.754822754591364
Local Utility: 0.9693463256191636
Per Feature Utility: {'LIMIT_BAL': np.float64(0.8136051565047032), 'SEX': 0.9454308093994779, 'EDUCATION': 0.870048019207683, 'MARRIAGE': 0.8938053097345132, 'AGE': np.float64(0.8117390314752414), 'PAY_0': 0.9771894093686354, 'PAY_2': 1.0517506733358986, 'PAY_3': 0.977544634640162, 'PAY_4': 0.996434603115031, 'PAY_5': 0.9834040199151761, 'PAY_6': 0.9714340125690344, 'BILL_AMT1': np.float64(0.715115168698903), 'BILL_AMT2': np.float64(0.6167119677746395), 'BILL_AMT3': np.float64(0.6874275380001131), 'BILL_AMT4': np.float64(0.6276146220206358), 'BILL_AMT5': np.float64(0.5930319783954827), 'BILL_AMT6': np.float64(0.6483992957071675), 'PAY_AMT1': np.float64(0.4743867400739974), 'PAY_AMT2': np.float64(0.6613371241072971), 'PAY_AMT3': np.float64(0.33873794861067225), 'PAY_AMT4': np.float64(0.3308403144831529), 'PAY_AMT5': np.float64(0.3450392886917327), 'PAY_AMT6': np.float64(0.8153721187442277), 'default': 0.9693463256191636